1.1 C D E
1.2 A
1.3 A B C
1.4 A
1.5 B
1.6 B

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import math

# 通用配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 模型定义
class MLP(nn.Module):
    """
    Two-layer fully connected MLP
    Input (flatten) -> Linear(784->256) -> Sigmoid/ReLU -> Linear(256->10)
    注意：输出直接给 CrossEntropyLoss，用的是 logits，不再加 Sigmoid/Softmax
    """
    def __init__(self, input_dim=784, hidden_dim=256, num_classes=10,
                 activation="sigmoid", use_bn=False):
        super().__init__()

        if activation == "sigmoid":
            act = nn.Sigmoid()
        else:
            act = nn.ReLU()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        if use_bn:
            layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(act)
        layers.append(nn.Linear(hidden_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (N, 1, 28, 28) or (N, 3, 32, 32)
        x = x.view(x.size(0), -1)  # flatten
        return self.net(x)


class SimpleCNNMNIST(nn.Module):
    """
    纯卷积 CNN for MNIST (1x28x28)
    结构（严格按作业）：
    Conv2d(1->16, k3,s1,p1) -> Sigmoid/ReLU -> MaxPool2d(2,2)
    Conv2d(16->32, k3,s1,p1) -> Sigmoid/ReLU -> MaxPool2d(2,2)
    Conv2d(32->10, k7,s1,p0) -> GAP -> (N,10)
    """
    def __init__(self, activation="sigmoid", use_bn=False):
        super().__init__()
        if activation == "sigmoid":
            act = nn.Sigmoid()
        else:
            act = nn.ReLU()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16) if use_bn else nn.Identity()

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(32) if use_bn else nn.Identity()

        # 输入 28x28，两个 2x2 池化后变成 7x7
        self.conv3 = nn.Conv2d(32, 10, kernel_size=7, stride=1, padding=0)

        self.act = act
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.pool(self.act(self.bn1(self.conv1(x))))
        x = self.pool(self.act(self.bn2(self.conv2(x))))
        x = self.conv3(x)           # (N,10,1,1)
        x = x.mean(dim=[2, 3])      # GAP -> (N,10)
        return x


class SimpleCNNCIFAR(nn.Module):
    """
    纯卷积 CNN for CIFAR-10 (3x32x32)
    结构与 MNIST 类似，只是第一层 in_channels=3，最后 kernel=8：
    Conv2d(3->16, k3,s1,p1) -> act -> pool
    Conv2d(16->32, k3,s1,p1) -> act -> pool
    Conv2d(32->10, k8,s1,p0) -> GAP -> (N,10)

    32x32 -> pool -> 16x16 -> pool -> 8x8 -> conv(k=8) -> 1x1
    """
    def __init__(self, activation="sigmoid", use_bn=False):
        super().__init__()
        if activation == "sigmoid":
            act = nn.Sigmoid()
        else:
            act = nn.ReLU()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16) if use_bn else nn.Identity()

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(32) if use_bn else nn.Identity()

        self.conv3 = nn.Conv2d(32, 10, kernel_size=8, stride=1, padding=0)

        self.act = act
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.pool(self.act(self.bn1(self.conv1(x))))
        x = self.pool(self.act(self.bn2(self.conv2(x))))
        x = self.conv3(x)
        x = x.mean(dim=[2, 3])  # GAP
        return x



# 数据加载
def get_mnist_dataloaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_ds = datasets.MNIST(root="./data", train=True,
                              download=True, transform=transform)
    test_ds = datasets.MNIST(root="./data", train=False,
                             download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size,
                             shuffle=False, num_workers=2)
    return train_loader, test_loader


def get_cifar10_dataloaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ])
    train_ds = datasets.CIFAR10(root="./data", train=True,
                                download=True, transform=transform)
    test_ds = datasets.CIFAR10(root="./data", train=False,
                               download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size,
                             shuffle=False, num_workers=2)
    return train_loader, test_loader


# 训练与评估
def evaluate(model, data_loader, criterion):
    model.eval()
    total = 0
    correct = 0
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc


def train(model, train_loader, test_loader,
          optimizer_name="sgd", lr=0.05, epochs=15):
    model.to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("Unsupported optimizer: " + optimizer_name)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0
        correct = 0
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        test_loss, test_acc = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"[Epoch {epoch:02d}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Test Acc: {test_acc:.4f}")

    return history


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


if __name__ == "__main__":
    batch_size = 64
    base_lr = 0.05
    epochs = 15

    # ========== 例 1：MNIST + MLP + Sigmoid + SGD（Baseline） ==========
    print("\n=== MNIST + MLP + Sigmoid + SGD (Baseline) ===")
    mnist_train, mnist_test = get_mnist_dataloaders(batch_size=batch_size)
    mlp_mnist = MLP(activation="sigmoid", use_bn=False)
    print("MLP(MNIST) trainable params:", count_trainable_params(mlp_mnist))
    hist_mlp_mnist_sgd = train(
        mlp_mnist,
        mnist_train,
        mnist_test,
        optimizer_name="sgd",
        lr=base_lr,
        epochs=epochs
    )

    # ========== 例 2：MNIST + CNN + Sigmoid + SGD（Baseline CNN） ==========
    print("\n=== MNIST + Pure CNN + Sigmoid + SGD (Baseline) ===")
    cnn_mnist = SimpleCNNMNIST(activation="sigmoid", use_bn=False)
    print("CNN(MNIST) trainable params:", count_trainable_params(cnn_mnist))
    hist_cnn_mnist_sgd = train(
        cnn_mnist,
        mnist_train,
        mnist_test,
        optimizer_name="sgd",
        lr=base_lr,
        epochs=epochs
    )

    # ========== 例 3：CIFAR-10 + MLP + Sigmoid + SGD ==========
    print("\n=== CIFAR-10 + MLP + Sigmoid + SGD ===")
    cifar_train, cifar_test = get_cifar10_dataloaders(batch_size=batch_size)
    # CIFAR-10 输入是 3x32x32 -> 3072
    mlp_cifar = MLP(input_dim=3 * 32 * 32, activation="sigmoid", use_bn=False)
    print("MLP(CIFAR-10) trainable params:", count_trainable_params(mlp_cifar))
    hist_mlp_cifar_sgd = train(
        mlp_cifar,
        cifar_train,
        cifar_test,
        optimizer_name="sgd",
        lr=base_lr,
        epochs=epochs
    )

    # ========== 例 4：CIFAR-10 + Pure CNN + Sigmoid + SGD ==========
    print("\n=== CIFAR-10 + Pure CNN + Sigmoid + SGD ===")
    cnn_cifar = SimpleCNNCIFAR(activation="sigmoid", use_bn=False)
    print("CNN(CIFAR-10) trainable params:", count_trainable_params(cnn_cifar))
    hist_cnn_cifar_sgd = train(
        cnn_cifar,
        cifar_train,
        cifar_test,
        optimizer_name="sgd",
        lr=base_lr,
        epochs=epochs
    )

    # 其他组合
    # 1) Sigmoid -> ReLU 对比（以 MNIST+MLP 为例）
    print("\n=== MNIST + MLP + ReLU + SGD ===")
    mlp_mnist_relu = MLP(activation="relu", use_bn=False)
    print("MLP(MNIST, ReLU) params:", count_trainable_params(mlp_mnist_relu))
    hist_mlp_mnist_relu_sgd = train(
        mlp_mnist_relu, mnist_train, mnist_test,
        optimizer_name="sgd", lr=base_lr, epochs=epochs
    )

    # 2) SGD -> Adam（以 MNIST+MLP+ReLU 为例）
    print("\n=== MNIST + MLP + ReLU + Adam ===")
    mlp_mnist_relu_adam = MLP(activation="relu", use_bn=False)
    hist_mlp_mnist_relu_adam = train(
        mlp_mnist_relu_adam, mnist_train, mnist_test,
        optimizer_name="adam", lr=1e-3, epochs=epochs
    )

    # 3) 加 BatchNorm（以 MNIST+MLP+ReLU 为例）
    print("\n=== MNIST + MLP + ReLU + BN + SGD ===")
    mlp_mnist_relu_bn = MLP(activation="relu", use_bn=True)
    hist_mlp_mnist_relu_bn_sgd = train(
        mlp_mnist_relu_bn, mnist_train, mnist_test,
        optimizer_name="sgd", lr=base_lr, epochs=epochs
    )


## Experimental Results and Analysis

In the baseline experiments, both MLP and pure CNN models were trained on the MNIST and CIFAR-10 datasets using the Sigmoid activation function and the SGD optimizer. The results demonstrate that convolutional networks outperform multilayer perceptrons across both datasets. This superiority arises from CNNs’ ability to capture local spatial features through convolutional kernels, whereas MLPs operate on flattened pixel inputs and thus lose spatial correlations. The performance gap becomes particularly evident on the more complex CIFAR-10 dataset, indicating that model architecture plays a crucial role in feature extraction and classification capability.

When the activation function was changed from Sigmoid to ReLU, the training process converged faster, and the final accuracy improved. This enhancement stems from the ReLU function’s non-saturating property, which helps preserve gradient magnitudes and mitigates the vanishing gradient problem commonly encountered with Sigmoid activations. As a result, models with ReLU not only train more efficiently but also achieve better generalization.

Replacing the optimizer from SGD to Adam led to a noticeably faster convergence during the early epochs. Adam’s adaptive learning rate mechanism—based on the first and second moments of the gradient—allows each parameter to adjust its update scale dynamically. This results in smoother optimization and better initial progress. However, if the learning rate is not properly tuned, Adam may introduce oscillations or suboptimal convergence. In general, Adam is advantageous for complex problems requiring rapid convergence, while SGD remains more stable over longer training horizons.

After introducing Batch Normalization layers, the training became more stable, with smoother loss curves and more consistent convergence behavior. BatchNorm normalizes intermediate activations, reducing internal covariate shift and ensuring more uniform gradient flow. It also introduces a mild regularization effect that mitigates overfitting, improving test performance. The normalized distributions of hidden activations became more compact and centered, confirming BatchNorm’s stabilizing influence on network dynamics.

Overall, CNN models achieved better accuracy, faster convergence, and stronger generalization than MLPs, particularly for high-dimensional image data. Combining ReLU activations and Batch Normalization proved to be an effective strategy for enhancing training stability and performance. Between the two optimizers, Adam provided faster early-stage learning, while SGD offered steady long-term convergence. In summary, architectural design, activation choice, and normalization techniques collectively determine the effectiveness and efficiency of deep neural network training.
